# Session 13 — 1/3: setup, correctness, baseline

Trains the ~20M model for 50M tokens **without** reversibility, at the largest
batch this GPU can hold. That batch is then reused unchanged by notebook 2 so
the arms are comparable.

In [1]:
!nvidia-smi
!git clone https://github.com/rjvim/era-v5-session13-reversibility repo 2>/dev/null || (cd repo && git pull)
%cd repo
!pip -q install -r requirements.txt
import sys; sys.path.insert(0, 'src')

Wed Sep 23 18:11:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   66C    P8             16W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Correctness first

The custom activation-free backward is checked against ordinary autograd in float64 before any timing number is trusted.

In [2]:
!pytest tests/test_reversibility.py -q

.........                                                                [100%]
9 passed in 8.19s


## Data — 50M training tokens (+1M val), GPT-2 encoding

Written once as a uint16 memmap so every arm reads identical tokens in identical order.

In [3]:
!python src/data.py --out_dir data --tokens 50000000
!ls -la data/

README.md: 100% 26.4k/26.4k [00:00<00:00, 58.1MB/s]
Resolving data files: 100% 2410/2410 [00:00<00:00, 24530.24it/s]
{'n_train': 50000000, 'n_val': 1000000, 'dataset': 'HuggingFaceFW/fineweb-edu', 'config': 'sample-10BT', 'encoding': 'gpt2'}
total 99628
drwxr-xr-x  2 root root      4096 Sep 23 18:13 .
drwxr-xr-x 12 root root      4096 Sep 23 18:12 ..
-rw-r--r--  1 root root       136 Sep 23 18:13 meta.json
-rw-r--r--  1 root root 100000000 Sep 23 18:13 train.bin
-rw-r--r--  1 root root   2000000 Sep 23 18:13 val.bin


## Largest batch the baseline can hold

Doubling + binary search over a *full* train step (fwd, bwd, optimiser), not a forward pass.

In [4]:
!python src/train.py --mode baseline --find_max_batch --seq_len 512

  batch 1: ok peak 0.62GB
  batch 2: ok peak 0.96GB
  batch 4: ok peak 1.62GB
  batch 8: ok peak 2.95GB
  batch 16: ok peak 5.61GB
  batch 32: ok peak 10.93GB
  batch 64: OOM
  batch 48: ok peak 16.25GB
  batch 56: OOM
  batch 52: OOM
  batch 50: ok peak 16.94GB
  batch 51: OOM
{
  "mode": "baseline",
  "seq_len": 512,
  "max_batch": 50,
  "peak_mem_gb_at_max": 16.94042205810547,
  "device": "NVIDIA L4",
  "dtype": "bf16",
  "h": 0.5
}


## Run 1 — baseline @ fixed batch

In [5]:
import json
BATCH_FIX = json.load(open('results/maxbatch_baseline.json'))['max_batch']
print('fixed batch =', BATCH_FIX)
!python src/train.py --mode baseline --batch_size {BATCH_FIX}     --run_name baseline_fixed --seq_len 512 --total_tokens 50000000 --resume


fixed batch = 50
step      0/1953 loss 10.8773 lr 1.54e-05 37,098 tok/s peak 16.78GB
step     50/1953 loss 7.2944 lr 6.00e-04 79,456 tok/s peak 16.94GB
step    100/1953 loss 6.9955 lr 5.99e-04 80,511 tok/s peak 16.94GB
step    150/1953 loss 6.6581 lr 5.96e-04 80,303 tok/s peak 16.94GB
step    200/1953 loss 6.6941 lr 5.91e-04 81,087 tok/s peak 16.94GB
step    250/1953 loss 6.4696 lr 5.84e-04 81,154 tok/s peak 16.94GB
step    300/1953 loss 6.4443 lr 5.76e-04 81,601 tok/s peak 16.94GB
step    350/1953 loss 6.3046 lr 5.66e-04 81,448 tok/s peak 16.94GB
step    400/1953 loss 6.3337 lr 5.54e-04 81,655 tok/s peak 16.94GB
step    450/1953 loss 6.1753 lr 5.41e-04 81,506 tok/s peak 16.94GB
step    500/1953 loss 6.3042 lr 5.26e-04 81,700 tok/s peak 16.94GB
step    550/1953 loss 6.1131 lr 5.10e-04 81,645 tok/s peak 16.94GB
step    600/1953 loss 5.9456 lr 4.93e-04 81,800 tok/s peak 16.94GB
step    650/1953 loss 5.9942 lr 4.75e-04 81,753 tok/s peak 16.94GB
step    700/1953 loss 5.9393 lr 4.56e-04 81,

In [6]:
d = json.load(open('results/baseline_fixed.json'))
print(f"loss {d['final_train_loss']:.4f} | val {d['final_val_loss']:.4f} | "
      f"{d['tok_per_s']:,.0f} tok/s | peak {d['peak_mem_gb']:.2f} GB")

loss 5.4043 | val 5.4307 | 81,954 tok/s | peak 16.94 GB


Carry `BATCH_FIX` into notebook 2 unchanged.